# Rewriter baseline on a Colab L4

Select **Runtime > Change runtime type > L4 GPU**. This notebook runs the shared diagnostic calibration set. It does not train a model. Human labels and judge calibration are pending. Use Colab Secrets for `HF_TOKEN` and, only for hosted calls, `OPENROUTER_API_KEY`.

The package wheel, config, and shared bundle come from your private Hugging Face dataset. The source revision and package hashes are recorded. Colab's CUDA-enabled PyTorch is retained; its actual version is recorded, so this is a separate run from the Modal baseline.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface-hub==1.33.0"], check=True
)
from huggingface_hub import HfApi, hf_hub_download

REPO = "jayshah5696/finetuning-labs-rewriter"
# Pin this to the runtime-assets revision recorded in the repository's release receipt.
REVISION = None
REVISION = HfApi().dataset_info(REPO, revision=REVISION).sha
assets_path = hf_hub_download(REPO, "runtime/assets.json", repo_type="dataset", revision=REVISION)
assets = json.loads(Path(assets_path).read_text())
paths = {}
for name, expected in assets["files"].items():
    p = Path(hf_hub_download(REPO, name, repo_type="dataset", revision=REVISION))
    assert hashlib.sha256(p.read_bytes()).hexdigest() == expected, name
    paths[name] = p
wheel = next(p for name, p in paths.items() if name.endswith(".whl"))
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        str(wheel),
        "transformers==5.21.0",
        "jinja2==3.1.6",
    ],
    check=True,
)
print({"source_commit": assets["code_commit"], "wheel": wheel.name})

In [ ]:
import tomllib

import torch

assert torch.cuda.is_available(), "Choose an L4 GPU runtime first."
assert "L4" in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
from finetuning_labs.rewriter.benchmark import audit, load_bundle
from finetuning_labs.rewriter.evaluation import generate_cases, judge_cases, liquid_backend

settings = tomllib.loads(paths["runtime/rewriter.toml"].read_text())
bundle = load_bundle()
assert audit(bundle)["case_count"] == 68
RUN = Path("/content/rewriter-baseline")
config = settings["liquid"] | {
    "code_commit": assets["code_commit"],
    "execution": "google-colab-L4",
    "dataset_revision": REVISION,
    "runtime_assets_sha256": assets["files"],
    "torch_version": torch.__version__,
}
print(
    {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cases": len(bundle.cases)}
)

In [ ]:
results = generate_cases(bundle, RUN / "liquid", config, liquid_backend(config))
from collections import Counter

print(Counter(row["status"] for row in results))
# Save the entire run directory before disconnecting. Existing calls are never silently rerun.

The next cell makes paid OpenRouter calls only when `RUN_HOSTED` is set to `True`. It uses the requested Luna baseline and Sol judge. Scores are diagnostic until compared with independent human labels. Keep this run's baseline and tuned comparison under identical runtime settings.

In [ ]:
RUN_HOSTED = False
if RUN_HOSTED:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "openai==2.54.0",
            "arka @ git+https://github.com/jayshah5696/arka.git@779040204b0f392465b7e6240bf9d9200816dc5d",
        ],
        check=True,
    )
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    from finetuning_labs.rewriter.evaluation import hosted_backend

    luna = settings["luna"] | {
        "code_commit": assets["code_commit"],
        "execution": "google-colab-openrouter",
    }
    generate_cases(bundle, RUN / "luna", luna, hosted_backend(luna), workers=4)
    rubric_path = Path(
        hf_hub_download(
            REPO, "bundle/supporting/judge-system.txt", repo_type="dataset", revision=REVISION
        )
    )
    judge_config = settings["judge"] | {"rubric": rubric_path.read_text()}
    for model in ["liquid", "luna"]:
        judge_cases(
            bundle,
            RUN / model,
            RUN / (model + "-judge"),
            judge_config,
            hosted_backend(judge_config, judge=True),
            workers=4,
        )
    print("Diagnostic metrics are under the two judge directories.")

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/rewriter-baseline", "zip", RUN)
files.download(archive)